# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [1]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [2]:
# Load the dataset from Phase 2 (update the path as needed)
DATA_PATH = r"D:\Churn Prediction And Analysis Project\Cell2Cell Data\cell2celltrain.csv"

df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")

df.head().T

Loaded dataset: 51047 rows x 58 columns


,0,1,2,3,4
CustomerID,3000002,3000010,3000014,3000022,3000026
Churn,Yes,Yes,No,No,Yes
MonthlyRevenue,24.0,16.99,38.0,82.28,17.14
MonthlyMinutes,219.0,10.0,8.0,1312.0,0.0
TotalRecurringCharge,22.0,17.0,38.0,75.0,17.0
DirectorAssistedCalls,0.25,0.0,0.0,1.24,0.0
OverageMinutes,0.0,0.0,0.0,0.0,0.0
RoamingCalls,0.0,0.0,0.0,0.0,0.0
PercChangeMinutes,-157.0,-4.0,-2.0,157.0,0.0
PercChangeRevenues,-19.0,0.0,0.0,8.1,-0.2


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [3]:
# TODO: Select the relevant columns and rows for your analysis.

columns_to_keep = df.columns.tolist()

columns_to_drop = [
    'CustomerID',
    'DroppedBlockedCalls',
    'NotNewCellphoneUser'
]
drop_reason = {
    'CustomerID': 'Unique identifier and does not contribute to churn prediction.',
    'DroppedBlockedCalls': 'Derived from DroppedCalls and BlockedCalls.',
    'NotNewCellphoneUser': 'Contains opposite information of NewCellphoneUser.'
}
df_selected = df.drop(
    columns=columns_to_drop,
    errors='ignore'
)
print(f"Original Shape: {df.shape}")
print(f"Selected Shape: {df_selected.shape}")
print("\nDropped Features:")

for col in columns_to_drop:
    print(f"- {col}: {drop_reason[col]}")

Original Shape: (51047, 58)
Selected Shape: (51047, 55)

Dropped Features:
- CustomerID: Unique identifier and does not contribute to churn prediction.
- DroppedBlockedCalls: Derived from DroppedCalls and BlockedCalls.
- NotNewCellphoneUser: Contains opposite information of NewCellphoneUser.


In [4]:
df_selected.head().T

,0,1,2,3,4
Churn,Yes,Yes,No,No,Yes
MonthlyRevenue,24.0,16.99,38.0,82.28,17.14
MonthlyMinutes,219.0,10.0,8.0,1312.0,0.0
TotalRecurringCharge,22.0,17.0,38.0,75.0,17.0
DirectorAssistedCalls,0.25,0.0,0.0,1.24,0.0
OverageMinutes,0.0,0.0,0.0,0.0,0.0
RoamingCalls,0.0,0.0,0.0,0.0,0.0
PercChangeMinutes,-157.0,-4.0,-2.0,157.0,0.0
PercChangeRevenues,-19.0,0.0,0.0,8.1,-0.2
DroppedCalls,0.7,0.3,0.0,52.0,0.0


In [5]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition

# df_selected = df_selected[df_selected['some_column'].notna()]
# print(f"Shape after row selection: {df_selected.shape}")

---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [6]:
# Handle Missing Values

# Create a clean copy
df_clean = df_selected.copy()

# Check Missing Values
missing = df_clean.isnull().sum().sort_values(ascending=False)

print("\nMissing Values Count:\n")
print(missing[missing > 0])

# Calculate Missing Percentage
missing_percent = (df_clean.isnull().mean()*100).sort_values(ascending=False)

print("\nMissing Percentage:\n")
print(missing_percent[missing_percent > 0])

# Remove Columns With More Than 50% Missing Values
cols_to_drop = missing_percent[missing_percent > 50].index

df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')

print("\nDropped Columns:")
print(list(cols_to_drop))

# Fill Numerical Columns using Median (better with outliers)
num_cols = df_clean.select_dtypes(include='number').columns

for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())


# Fill Categorical Columns Using Mode
cat_cols = df_clean.select_dtypes(include='object').columns

for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Final Validation
print("\nFinal Dataset Shape:")
print(df_clean.shape)

print("\nRemaining Missing Values:")
print(df_clean.isnull().sum().sum())


Missing Values Count:

AgeHH2                   909
AgeHH1                   909
PercChangeMinutes        367
PercChangeRevenues       367
MonthlyMinutes           156
TotalRecurringCharge     156
DirectorAssistedCalls    156
OverageMinutes           156
RoamingCalls             156
MonthlyRevenue           156
ServiceArea               24
CurrentEquipmentDays       1
HandsetModels              1
Handsets                   1
dtype: int64

Missing Percentage:

AgeHH2                   1.780712
AgeHH1                   1.780712
PercChangeMinutes        0.718945
PercChangeRevenues       0.718945
MonthlyMinutes           0.305601
TotalRecurringCharge     0.305601
DirectorAssistedCalls    0.305601
OverageMinutes           0.305601
RoamingCalls             0.305601
MonthlyRevenue           0.305601
ServiceArea              0.047015
CurrentEquipmentDays     0.001959
HandsetModels            0.001959
Handsets                 0.001959
dtype: float64

Dropped Columns:
[]

Final Dataset Shape:
(

In [7]:
# Remove duplicate records.

before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"Removed {before - after} duplicate rows. Remaining: {after} rows.")

Removed 0 duplicate rows. Remaining: 51047 rows.


In [8]:
# Correct Invalid Values (Negative)

invalid_cols = [
'MonthlyRevenue',
'TotalRecurringCharge',
'CurrentEquipmentDays'
]
for col in invalid_cols:
    print(col, ":", (df_clean[col] < 0).sum())

# Replace Negative Values
for col in invalid_cols:
    df_clean[col] = df_clean[col].clip(lower=0)

# Validation
print("\nRemaining Invalid Values:")

for col in invalid_cols:
    print(col, ":", (df_clean[col] < 0).sum())

MonthlyRevenue : 3
TotalRecurringCharge : 8
CurrentEquipmentDays : 76

Remaining Invalid Values:
MonthlyRevenue : 0
TotalRecurringCharge : 0
CurrentEquipmentDays : 0


In [9]:
# Handle outliers.

# Detect Columns Containing Outliers
num_cols = df_clean.select_dtypes(include='number').columns

outlier_cols = []

for col in num_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = (
        (df_clean[col] < lower) |
        (df_clean[col] > upper)
    ).sum()
    if outliers > 0:
        outlier_cols.append([col, outliers])

print("\nColumns With Outliers:\n")

for col, count in outlier_cols:
    print(f"{col}: {count}")


Columns With Outliers:

MonthlyRevenue: 3009
MonthlyMinutes: 2588
TotalRecurringCharge: 824
DirectorAssistedCalls: 5530
OverageMinutes: 5980
RoamingCalls: 10070
PercChangeMinutes: 6926
PercChangeRevenues: 13471
DroppedCalls: 3712
BlockedCalls: 5517
UnansweredCalls: 3630
CustomerCareCalls: 6721
ThreewayCalls: 4622
ReceivedCalls: 3641
OutboundCalls: 3342
InboundCalls: 4973
PeakCallsInOut: 2803
OffPeakCallsInOut: 3624
CallForwardingCalls: 234
CallWaitingCalls: 7448
MonthsInService: 1218
UniqueSubs: 1874
ActiveSubs: 611
Handsets: 4414
HandsetModels: 2008
CurrentEquipmentDays: 1446
RetentionCalls: 1745
RetentionOffersAccepted: 881
ReferralsMadeBySubscriber: 2384
AdjustmentsToCreditRating: 1838


In [10]:
# Handle Selected Outliers Using IQR Capping

outlier_cols = [
'DirectorAssistedCalls',
'ThreewayCalls',
'CallForwardingCalls',
'CallWaitingCalls',
'UniqueSubs',
'ActiveSubs',
'Handsets',
'HandsetModels',
'CurrentEquipmentDays',
'RetentionOffersAccepted',
'ReferralsMadeBySubscriber',
'AdjustmentsToCreditRating'
]

for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean[col] = (
        df_clean[col]
        .clip(
            lower,
            upper
        )
    )

print("Selected outliers treated.")

Selected outliers treated.


In [11]:
# Validation

remaining_outliers = []
for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = (
        (df_clean[col] < lower) |
        (df_clean[col] > upper)
    ).sum()
    remaining_outliers.append([col, outliers])

print(remaining_outliers)

[['DirectorAssistedCalls', 0], ['ThreewayCalls', 0], ['CallForwardingCalls', 0], ['CallWaitingCalls', 0], ['UniqueSubs', 0], ['ActiveSubs', 0], ['Handsets', 0], ['HandsetModels', 0], ['CurrentEquipmentDays', 0], ['RetentionOffersAccepted', 0], ['ReferralsMadeBySubscriber', 0], ['AdjustmentsToCreditRating', 0]]


In [12]:
"""
Cleaning Summary:

Missing values were identified in several numerical and categorical features. Since all missing percentages were below 2%, no columns were removed. Numerical missing values were imputed using the median, while categorical variables were filled using the mode to preserve data distribution and avoid bias.

Duplicate records were checked, and no duplicate rows were found in the dataset.

Invalid negative values were detected in MonthlyRevenue, TotalRecurringCharge, and CurrentEquipmentDays. Since negative values are not meaningful for these business-related features, they were corrected by replacing them with zero. A final validation confirmed that no invalid values remained.

Outliers were detected across multiple numerical features using the Interquartile Range (IQR) method. A subset of features with significant outliers was treated using IQR-based capping (winsorization) to reduce the effect of extreme values while preserving data integrity and avoiding information loss.

Final validation confirmed that the dataset contains no missing values, no duplicate records, no invalid negative values in the selected columns, and no remaining outliers in the treated features.

The final cleaned dataset is now consistent, reliable, and ready for further feature engineering and modeling.
"""

'\nCleaning Summary:\n\nMissing values were identified in several numerical and categorical features. Since all missing percentages were below 2%, no columns were removed. Numerical missing values were imputed using the median, while categorical variables were filled using the mode to preserve data distribution and avoid bias.\n\nDuplicate records were checked, and no duplicate rows were found in the dataset.\n\nInvalid negative values were detected in MonthlyRevenue, TotalRecurringCharge, and CurrentEquipmentDays. Since negative values are not meaningful for these business-related features, they were corrected by replacing them with zero. A final validation confirmed that no invalid values remained.\n\nOutliers were detected across multiple numerical features using the Interquartile Range (IQR) method. A subset of features with significant outliers was treated using IQR-based capping (winsorization) to reduce the effect of extreme values while preserving data integrity and avoiding in

---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [ ]:
# Create derived attributes / new features.

df_fe=df_clean.copy()

# Total Call Activity نشاط العميل كله في المكالمات
df_fe['TotalCalls']=df_fe['OutboundCalls']+df_fe['InboundCalls']+df_fe['CustomerCareCalls']+df_fe['ThreewayCalls']

# Average Revenue per Minute العميل بيدفع كام مقابل الدقيقة
df_fe['RevenuePerMinute']=df_fe['MonthlyRevenue']/(df_fe['MonthlyMinutes']+1)

# Call Efficiency العميل بيعمل كام مكالمة بالنسبة لمدة استخدامه.
df_fe['CallsPerMinute']=df_fe['TotalCalls']/(df_fe['MonthlyMinutes']+1)

# Equipment Usage Ratio  متوسط الاستخدام لكل جهاز
df_fe['UsagePerDevice']=df_fe['MonthlyMinutes']/(df_fe['Handsets']+1)

In [ ]:
# Binning 
df_fe['TenureGroup'] = pd.cut( # مدة بقاء العميل 
    df_fe['MonthsInService'],
    bins=[0, 12, 24, 48, 100],
    labels=['New', 'Medium', 'Long', 'Very Long']
)

df_fe['AgeGroup_HH1'] = pd.cut(
    df_fe['AgeHH1'],
    bins=[0, 30, 45, 60, 100],
    labels=['Young', 'Adult', 'MiddleAge', 'Senior']
)

df_fe['AgeGroup_HH2'] = pd.cut(
    df_fe['AgeHH2'],
    bins=[0, 30, 45, 60, 100],
    labels=['Young', 'Adult', 'MiddleAge', 'Senior']
)


In [15]:
df_fe['TenureGroup'] = df_fe['TenureGroup'].map({
    'New': 0,
    'Medium': 1,
    'Long': 2,
    'Very Long': 3
})

df_fe['AgeGroup_HH1'] = df_fe['AgeGroup_HH1'].map({
    'Young': 0,
    'Adult': 1,
    'MiddleAge': 2,
    'Senior': 3
})

df_fe['AgeGroup_HH2'] = df_fe['AgeGroup_HH2'].map({
    'Young': 0,
    'Adult': 1,
    'MiddleAge': 2,
    'Senior': 3
})

In [ ]:
df_fe = df_fe.copy()
#Encoding
cat_cols = df_fe.select_dtypes(include='object').columns

# Clean text
for col in cat_cols:
    df_fe[col] = (
        df_fe[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace('nan', 'unknown')
    )

# Encode
for col in cat_cols:
    unique_vals = df_fe[col].unique()

    # Binary yes/no
    if set(unique_vals).issubset({'yes', 'no'}):
        df_fe[col] = df_fe[col].map({'no': 0, 'yes': 1})

    # yes/no/unknown
    elif set(unique_vals).issubset({'yes', 'no', 'unknown'}):
        df_fe[col] = df_fe[col].map({'no': 0, 'unknown': 1, 'yes': 2})

    # known/unknown
    elif set(unique_vals).issubset({'known', 'unknown'}):
        df_fe[col] = df_fe[col].map({'unknown': 0, 'known': 1})

    # ordinal
    elif set(unique_vals).issubset({'good', 'medium', 'high'}):
        df_fe[col] = df_fe[col].map({'good': 0, 'medium': 1, 'high': 2})

    # default → one-hot
    else:
        df_fe = pd.get_dummies(df_fe, columns=[col], drop_first=True)



In [ ]:
# numerical columns only
num_cols = df_fe.select_dtypes(include=['int64', 'float64']).columns

# remove target if present
num_cols = num_cols.drop('Churn', errors='ignore')
#Scaling
scaler = StandardScaler()

df_fe[num_cols] = scaler.fit_transform(df_fe[num_cols])

---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [18]:
# Since data is from a single source, no merging is required

---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [19]:
# Apply final formatting — data types, column order, renaming.

df_final = df_fe.copy()
#Convert bool
bool_cols = df_final.select_dtypes(include='bool').columns
df_final[bool_cols] = df_final[bool_cols].astype(int)

#convert category
cat_cols = df_final.select_dtypes(include='category').columns
df_final[cat_cols] = df_final[cat_cols].apply(lambda x: x.cat.codes)

df_final['Churn'] = df_final['Churn'].astype(int)
#Column reordering (Target last)
cols = [c for c in df_final.columns if c != 'Churn'] + ['Churn']
df_final = df_final[cols]


In [20]:
print("Shape:", df_final.shape)
print("Missing values:", df_final.isnull().sum().sum())
print(df_final.dtypes.value_counts())

Shape: (51047, 834)
Missing values: 0
int32      778
float64     53
int8         3
Name: count, dtype: int64


In [21]:
df_final.to_csv("final_churn_dataset.csv", index=False)

In [22]:
#Save the prepared dataset for use in Phase 4 (Modelling).

OUTPUT_PATH = r"D:\Churn Prediction And Analysis Project\Cell2Cell Data\churn_final_dataset.csv"

df_final.to_csv(OUTPUT_PATH, index=False)

print(f"Prepared dataset saved to: {OUTPUT_PATH}")

Prepared dataset saved to: D:\Churn Prediction And Analysis Project\Cell2Cell Data\churn_final_dataset.csv
